Setup da Camada Bronze

In [0]:
from pyspark.sql.functions import current_timestamp
import requests

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

caminho_volume = "/Volumes/workspace/default/raw_data/"

Ingestão de Arquivos CSV

In [0]:
# Mapeamento de Arquivos para Tabelas
tabelas_csv = {
    "olist_customers_dataset.csv": "bronze.tb_customers",
    "olist_geolocation_dataset.csv": "bronze.tb_geolocalizacao",
    "olist_order_items_dataset.csv": "bronze.tb_order_items",
    "olist_order_payments_dataset.csv": "bronze.tb_order_payments",
    "olist_order_reviews_dataset.csv": "bronze.tb_order_reviews",
    "olist_orders_dataset.csv": "bronze.tb_orders",
    "olist_products_dataset.csv": "bronze.tb_products",
    "olist_sellers_dataset.csv": "bronze.tb_sellers",
    "product_category_name_translation.csv": "bronze.tb_product_category_name_translation"
}

for arquivo, tabela in tabelas_csv.items():
    caminho_arquivo = f"{caminho_volume}{arquivo}"
    
    df = spark.read.csv(caminho_arquivo, header=True, inferSchema=True)
    df = df.withColumn("timestamp_ingestion", current_timestamp())
    
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"✅ Tabela {tabela} ingerida com sucesso!")

✅ Tabela bronze.tb_customers ingerida com sucesso!
✅ Tabela bronze.tb_geolocalizacao ingerida com sucesso!
✅ Tabela bronze.tb_order_items ingerida com sucesso!
✅ Tabela bronze.tb_order_payments ingerida com sucesso!
✅ Tabela bronze.tb_order_reviews ingerida com sucesso!
✅ Tabela bronze.tb_orders ingerida com sucesso!
✅ Tabela bronze.tb_products ingerida com sucesso!
✅ Tabela bronze.tb_sellers ingerida com sucesso!
✅ Tabela bronze.tb_product_category_name_translation ingerida com sucesso!


Cotação Dolar

In [0]:
# Configuração da Janela de Tempo
data_inicio = "09-01-2016" 
data_fim = "10-31-2018"

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)

if response.status_code == 200:
    dados_api = response.json()['value']
    df_cotacao = spark.createDataFrame(dados_api)
    df_cotacao = df_cotacao.withColumn("timestamp_ingestion", current_timestamp())
    
    df_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.tb_cotacao_dolar")
    print("✅ Tabela bronze.tb_cotacao_dolar ingerida com sucesso!")
else:
    raise Exception(f"❌ Erro ao acessar a API do BCB: Status Code {response.status_code}")

✅ Tabela bronze.tb_cotacao_dolar ingerida com sucesso!
